# Measuring Deviation

This notebook measures FM deviation from either a local tone test capture or a synthetic fallback. The main metric is the instantaneous frequency swing around the carrier.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from rf_utils import *
from IPython.display import Audio, Markdown, display
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

%matplotlib widget

plt.rcParams.update({
    "figure.figsize": (12, 4),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})


In [ ]:
CAPTURE_ROOT = ROOT / "assets" / "local"
capture_status = probe_rtlsdr()
display(Markdown(
    f"**RTL-SDR status:** installed={capture_status['installed']}, "
    f"available={capture_status['available']}. {capture_status['message']}"
))


In [ ]:
capture_path = CAPTURE_ROOT / "deviation_test_iq.npz"
tone_freq = 1000

if capture_path.exists():
    fs_iq, iq = load_complex_capture(capture_path)
    print(f"Loaded local capture: {capture_path.name}, fs={fs_iq}")
else:
    fs_iq = 240_000
    t = np.arange(0, 2.0, 1 / fs_iq)
    tone = np.cos(2 * np.pi * tone_freq * t)
    iq = synthesize_fm_iq(tone, fs=fs_iq, carrier_offset=0, freq_dev=2500)
    print("Using synthetic FM tone fallback.")


In [ ]:
inst_freq = np.angle(iq[1:] * np.conj(iq[:-1])) * fs_iq / (2 * np.pi)
inst_freq = np.concatenate([inst_freq, inst_freq[-1:]])
inst_freq = inst_freq - np.mean(inst_freq)
peak_dev = np.max(np.abs(inst_freq))
rms_dev = np.sqrt(np.mean(inst_freq**2))

fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
plot_waveform(inst_freq[:15000], fs=fs_iq, ax=axes[0], title="Instantaneous frequency")
plot_spectrum(inst_freq, fs=fs_iq, ax=axes[1], title="Deviation spectrum")
axes[1].set_xlim(0, 5000)
axes[1].set_ylim(-100, 5)
plt.tight_layout()

display(Markdown(f"**Peak deviation:** {peak_dev:.1f} Hz"))
display(Markdown(f"**RMS deviation:** {rms_dev:.1f} Hz"))


## Key Takeaway

Deviation is directly measurable from phase progression in IQ data. You do not need a special black-box meter if you can compute instantaneous frequency robustly.